# TRAINING of Hyperparameters
### Three parameter as input case
This code as the purpose to find the best hyperparameters in the case 2 step. In particular, it is worth noticing that the first NN is already optimized. 

In [14]:
#########################     LIBRARIES     ##########################
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam,Nadam,Adamax
from ann_functions3D import getModel, kCrossVal, transfBestparam
from time import perf_counter
import pandas

seed = 7
np.random.seed(seed)

In [15]:
########################     PREPARATION      ##########################
HF_data = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/HF_num_data.txt").astype(int) #list of number of HF data
HF_data_str = [str(num) for num in HF_data]
Nlf_models = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/N_bases.txt").astype(int)   #list of number of bases
U_train_LF_full = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Ulf_train.txt")

n_bases = U_train_LF_full.shape[1]
m = list(Nlf_models).index(n_bases)
n_HF = HF_data_str[-1]  
n_HF_txt = str(n_HF) + '.txt'

In [16]:
#########################     TRAIN SET      ##########################
mu_train_LF = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/mu_train_LF.txt")
Nlf = np.size(mu_train_LF[:,0])  #number of low-fidelity data to TRAIN the NN

U_lf_train_full = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Ulf_train.txt")
U_lf_train = U_lf_train_full[:,m]

mu_train_HF = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/mu_train_HF_" + n_HF_txt)
Nhf = np.size(mu_train_HF)  # number of high-fidelity data to TRAIN the NN

# ADD NOISE

permutation = np.random.permutation(len(mu_train_LF))
mu_train_LF=mu_train_LF[permutation,:][0:10,:]

noise_stddev = np.mean(mu_train_LF,axis=0)*0.1
noise = np.random.normal(0, noise_stddev, np.shape(mu_train_LF))
mu_train_LF=mu_train_LF+noise 

Nlf = np.size(mu_train_LF)  #number of low-fidelity data to TRAIN the NN

U_lf_train_full=U_lf_train_full[permutation][0:10,:]

noise_stddev = 0.005
noise = np.random.normal(0, noise_stddev, np.shape(U_lf_train_full))
U_lf_train_full=U_lf_train_full+noise

U_hf_train = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Uhf_train_" + n_HF_txt)

NepoLF = 4000        # number of epochs for first NN: NN_LF
NepoLin = 1500       # number of epochs for second NN: NN_Lin
NepoHF = 3000        # number of epochs for third NN: NN_HF

In [17]:
#########################     TEST SET      ##########################
mu_test = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/mu_test.txt")
N_test = np.size(mu_test)

U_lf_test_full = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/Ulf_test.txt")
U_lf_test = U_lf_test_full[:,m]

U_hf_test = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/Uhf_test.txt")

In [18]:
########################     NORMALIZATION  #########################
# diverso da codice di esempio 5 params... da verificare

#Input
mu_max = np.max(mu_test)
mu_min = np.min(mu_test)

mu_test_norm = (mu_test - mu_min) / (mu_max - mu_min)
mu_train_LF_norm = (mu_train_LF - mu_min) / (mu_max - mu_min)
mu_train_HF_norm = (mu_train_HF - mu_min) / (mu_max - mu_min)

#Output
# TRANSFORMATION     
# in order to reduce the linear relationship between the Young modulus and the displacement
#U_lf_train = np.exp(U_lf_train*100)*1e-6
#U_lf_test = np.exp(U_lf_test*100)*1e-6
#U_hf_train = np.exp(U_hf_train*100)*1e-6
#U_hf_test = np.exp(U_hf_test*100)*1e-6

###
lfmean = np.mean(U_lf_test)
hfmean = np.mean(U_hf_test)

U_lf_train = U_lf_train - lfmean
U_lf_test = U_lf_test - lfmean
U_hf_train = U_hf_train - hfmean
U_hf_test = U_hf_test - hfmean

In [19]:
##########################       FIRST NN: NN_LF     ##########################
K.clear_session()
bestLF_params = {'lr' : 0.0255, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam'}

modelLF = getModel(bestLF_params,'LF')
histLF = modelLF.fit(mu_train_LF_norm, U_lf_train ,epochs=NepoLF,batch_size=Nlf, verbose = 0)
print('LF NN done')

ULF = modelLF.predict(mu_test_norm)
print('\nLF Model:')

test_mse = np.mean(np.square(U_lf_test - ULF[:,0]))
print(f"Test MSE: {test_mse}")

r_2 = 1 - np.sum(np.square(U_lf_test - ULF[:,0])) / np.sum(np.square(U_lf_test - np.mean(U_lf_test)))
print(f"R^2: {r_2}")



print(f"\n-------  #HF data = {n_HF}  -------")
start = perf_counter()


ValueError: Data cardinality is ambiguous:
  x sizes: 10
  y sizes: 100
Make sure all arrays contain the same number of samples.

In [5]:
##########################    SECOND NN: NN_Lin    ##########################
hf_help = modelLF.predict(mu_train_HF_norm)
hf_lin = np.append(mu_train_HF_norm, hf_help,axis=1)

K.clear_session()

test_help = modelLF.predict(mu_test_norm)

bestLin_params = {'lr' : 0.001, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam', 'l2weight' : 0.01}
modelLin = getModel(bestLin_params,'Hflin')
histLin = modelLin.fit(hf_lin, U_hf_train, epochs= NepoLin, batch_size = Nhf, verbose = 0)

ULin = modelLin.predict(np.append(mu_test_norm, test_help, axis = 1))



NameError: name 'modelLF' is not defined

In [6]:
##########################    THIRD NN: NN_HF    ##########################
#Input for training NN_HF
hf_help1 = modelLin.predict(hf_lin)
hf_final = np.append(hf_lin, hf_help1, axis=1)

#Input for testing NN_HF
test_help_1 = modelLin.predict(np.append(mu_test_norm, test_help, axis = 1))
test_input_in = np.append(mu_test_norm, np.append(test_help, test_help_1, axis=1), axis=1)

name = '3step'

NameError: name 'modelLin' is not defined

In [7]:
####################    HYPERPARAMETER OPTIMIZATION    #######################
MAX_EVAL = 15

K.clear_session()
bayes_trials = Trials()
opt_list = ['Adam','Adamax']
kernel_list = ['uniform','glorot_uniform']
aux_dic = {'opt': opt_list, 'kernel_init': kernel_list}
space = {
    'nodes' : hp.qloguniform('nodes',np.log(2),np.log(128),2),
     'l2weight' : hp.loguniform('l2weight',np.log(0.0001),np.log(0.1)),
     'lr': hp.loguniform('lr',np.log(0.0001),np.log(0.1)),
     'kernel_init': hp.choice('kernel_init',kernel_list),
     'opt' : hp.choice('opt',opt_list)}

p=3

def objective(params):
    K.clear_session()
    CVres = kCrossVal(p,Nhf,NepoHF,hf_final, U_hf_train,params,name)
    return {'loss' : CVres, 'params':params, 'status': STATUS_OK}

best_params = fmin(fn = objective,
                   space = space,
                   algo = tpe.suggest,
                   max_evals = MAX_EVAL,
                   trials = bayes_trials)
K.clear_session()
transfBestparam(best_params,aux_dic)
finalModel = getModel(best_params,name)

print(best_params)


  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

job exception: name 'NepoHF' is not defined



  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]


NameError: name 'NepoHF' is not defined

In [ ]:
####################    NN_HF training and PREDICTION    #######################
finalModel = getModel(best_params,name)
hist = finalModel.fit(hf_final,U_hf_train,validation_data=(test_input_in, U_hf_test),epochs=NepoHF, batch_size=Nhf, validation_freq=50, verbose=0)

UHF = finalModel.predict(test_input_in)

stop = perf_counter()
elapsed = stop - start
print('Elapsed time: ', elapsed)
print('\nHF Model:')

test_mse = np.mean(np.square(U_hf_test - UHF[:,0]))
print(f"Test MSE: {test_mse}")

r2_HF = 1 - np.sum(np.square(U_hf_test - UHF[:,0])) / np.sum(np.square(U_hf_test - np.mean(U_hf_test)))
print(f"R^2: {r2_HF}")

print('Number of base functions: ', int(Nlf_models[m]))
print('Number of HF data: ', n_HF)

In [ ]:
####################    TRAINING INSIGHTS    #######################
plt.figure()
plt.subplot(2,1,1)
plt.plot(hist.history['mse'],color='red',label='High fidelity train mse')
plt.plot(histLin.history['mse'],color='green',label='high fidelity lin')
plt.plot(histLF.history['mse'],color='black',label='Low fidelity')
plt.legend()
plt.yscale('log')
plt.subplot(2,1,2)
plt.plot(hist.history['val_mse'],color='red')
plt.yscale('log')
